# Formula-One Driver-Risk & Predictive-Maintenance Framework  
1.  Environment installs
2.  Global config + imports
3.  FastF1 session helpers
4.  Telemetry enrichment (positional context)
5.  Dataset builder (windows + labels)
6.  PyTorch Dataset / DataLoader
7. CNN-BiLSTM             |
8.  Temporal Transformer  - continuation of CNN/BiLSTM     | choose one
9.  Lightning wrapper + Trainer
10.  Training run
11. Metrics plots (ROC / PR / CM)
12. Lead-time histogram
13. Grad-CAM temporal saliency
14. ONNX export
15. FastAPI micro-service (+ Prometheus metrics)
16. Dockerfile + docker-compose (Driver-Risk / Prom / Grafana)
17. Predictive-Maintenance LSTM-VAE  (optional)

***Seth R. Stock***

In [1]:
import os, json, math, time, warnings, random, pathlib, itertools, glob
import numpy as np
import pandas as pd
from pathlib import Path
import fastf1, fastf1.plotting
from fastf1 import plotting
from fastf1 import utils as f1utils
from fastf1 import Cache
import os, fastf1

CACHE_DIR = "./cache_f1"
os.makedirs(CACHE_DIR, exist_ok=True)   # <-- no error if it already exists

fastf1.Cache.enable_cache(CACHE_DIR)     # FastF1 keeps local .pkl/feather files

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, confusion_matrix
from alive_progress import alive_bar
pl.seed_everything(42, workers=True)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\seths\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\seths\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\seths\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_lo

42

In [2]:
# 2 · CONFIG  ────────────────────────────────────────────────────────────
CFG = dict(
    SEASONS    = [2020, 2021, 2022, 2023, 2024],   # choose any range
    SESSIONS   = ["R"],                            # add "SQ", "Q" if desired
    WINDOW_MS  = 3000,
    STEP_MS    = 100,
    LOOKAHEAD  = 3000,          # ms  (label lead-time)
    BATCH      = 256,
    EPOCHS     = 10,
    LR         = 3e-4,
    DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
)

FEATURES = ["Throttle","Brake","Steer","Speed","nGear","RPM",
            "GapToLeader","Sector","sin_u","cos_u","DRS","RelSpeedAhead"]
TARGET   = "incident"


In [3]:
# 3 · SCHEDULE & SESSION HELPERS  ────────────────────────────────────────
from fastf1 import events

def safe_schedule(year: int) -> pd.DataFrame:
    """Resilient schedule fetch that tries two back-ends."""
    for be in [{"backend":"ergast", "force_ergast":True},
               {"backend":"fastf1"}]:
        try:
            return events.get_event_schedule(year, include_testing=False, **be)
        except Exception as e:
            print(f"   backend {be['backend']} failed → {e}")
    raise RuntimeError(f"No schedule available for {year}")

def fetch_session(year:int, gp:str, code:str="R"):
    ses = fastf1.get_session(year, gp, code)
    ses.load()                                     # hits cache or downloads
    tel_all = []
    for _, lap in ses.laps.iterrows():
        t = lap.get_telemetry()                    # 10 ms telemetry
        t['LapNumber'] = lap.LapNumber; t['Driver'] = lap.Driver
        tel_all.append(t)
    tel = pd.concat(tel_all, ignore_index=True)
    flags = ses.api.session_status                 # SC / VSC / Yellow
    return ses, tel, flags


In [4]:
# 4 · ENRICHMENT & NUMERIC COERCION  ─────────────────────────────────────
def _gap_table(laps):
    g = laps[['LapNumber','Driver','LapTime']].copy()
    g['LapSec'] = pd.to_timedelta(g.LapTime).dt.total_seconds().fillna(0.0)
    g['GapToLeader'] = g.groupby('LapNumber')['LapSec'].cumsum()
    return g[['LapNumber','Driver','GapToLeader']]

def enrich(tel, ses):
    tel = tel.merge(_gap_table(ses.laps), on=['LapNumber','Driver'], how='left')
    laplen = tel.groupby(['Driver','LapNumber'])['Distance'].transform('max')
    tel['Sector'] = pd.cut(tel.Distance/laplen, [0,.333,.666,1], labels=[1,2,3]).astype(int)
    tel['u'] = tel.Distance / laplen
    tel['sin_u'], tel['cos_u'] = np.sin(2*np.pi*tel.u), np.cos(2*np.pi*tel.u)
    tel['DRS'] = tel.get('DRS', 0)
    tel['RelSpeedAhead'] = tel.groupby('Date')['Speed'].diff().fillna(0)
    return tel

def _to_float(x):
    """Flatten lists/Timedeltas; return float32 or NaN."""
    if isinstance(x, (list, tuple)) and len(x):
        x = x[0]
    if isinstance(x, pd.Timedelta):
        x = x.total_seconds()
    try:
        return np.float32(x)
    except Exception:
        return np.nan

def engineer(tel, flags, ses):
    tel = enrich(tel, ses)
    for col in FEATURES:
        tel.setdefault(col, np.nan)

    df = tel[['Driver','Date',*FEATURES]].copy()
    # incident label
    win = pd.Timedelta(milliseconds=CFG['LOOKAHEAD'])
    df[TARGET] = 0
    for _, r in flags.iterrows():
        m = (df.Date >= r.Time - win) & (df.Date < r.Time)
        df.loc[m, TARGET] = 1

    df[FEATURES] = df[FEATURES].applymap(_to_float).fillna(0.0)
    return df


In [5]:
def build_windows(df):
    win, step = CFG['WINDOW_MS']//10, CFG['STEP_MS']//10
    X, y = [], []
    for drv in df.Driver.unique():
        sub = df[df.Driver == drv]
        if len(sub) < win:
            continue
        arr = sub[FEATURES].to_numpy(np.float32)
        labs = sub[TARGET].to_numpy(np.int8)

        Xw = fast_windows(arr, win, step)          # (N, win, F)
        yw = fast_windows(labs[:, None], win, step).max(axis=1).squeeze()

        X.append(Xw); y.append(yw)

    return np.concatenate(X) if X else [], np.concatenate(y) if y else []


In [ ]:
# %% [code] ###################################################################
# 6 · FAST SEASON LOAD  (parallel I/O + lazy dataset)

import numpy as np, os, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from torch.utils.data import Dataset, DataLoader

# ----------------------- (C) parallel single-session worker ---------------
def _one_session(year:int, gp:str, sess_code:str):
    """Download + window a single GP; returns (X, y) or (None, None) on failure."""
    try:
        ses, tel, flags = fetch_session(year, gp, sess_code)
        X, y = build_windows(engineer(tel, flags, ses))       # (N, win, F)
        if len(X):
            print(f"✓ {year} {gp:<28}  windows {len(X):,}")
            return X, y
        else:
            warnings.warn(f"    {gp} – no usable windows")
    except Exception as e:
        warnings.warn(f"    Skipped {gp}: {e}")
    return None, None

# ----------------------- dispatch work for all seasons --------------------
futures = []
max_workers = min(4, os.cpu_count()*2)      # threads → I/O bound
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    for yr in CFG['SEASONS']:
        sched = safe_schedule(yr)
        for code in CFG['SESSIONS']:
            for _, ev in sched.iterrows():
                futures.append(ex.submit(_one_session, yr, ev.EventName, code))

X_list, y_list = [], []
for fut in as_completed(futures):
    res = fut.result()
    if res[0] is not None:
        X_list.append(res[0]); y_list.append(res[1])

if not X_list:
    raise RuntimeError("No windows collected — check error log above")

# concatenate once at end (continuous block in RAM)
X_all = np.concatenate(X_list, axis=0)
y_all = np.concatenate(y_list, axis=0)
print(f"\nTotal windows across {CFG['SEASONS']}: {len(X_all):,}")

# ----------------------- (D) lazy dataset that shuffles by index ----------
perm  = np.random.permutation(len(X_all))
split = int(0.8 * len(perm))
train_idx, val_idx = perm[:split], perm[split:]

class RiskDS(Dataset):
    """Holds a reference to the big NumPy array; shuffles via index only."""
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return (torch.from_numpy(self.X[i]), torch.tensor(self.y[i]))

train_dl = DataLoader(
    RiskDS(X_all[train_idx], y_all[train_idx]),
    batch_size=CFG['BATCH'], shuffle=True, num_workers=10
)

val_dl = DataLoader(
    RiskDS(X_all[val_idx], y_all[val_idx]),
    batch_size=CFG['BATCH'], shuffle=False, num_workers=10
)

print(f"Train windows: {len(train_dl.dataset):,} | Val windows: {len(val_dl.dataset):,}")


d:\Python\Lib\site-packages\fastf1\events.py:498: UserWarning: Option ``force_ergast`` has been deprecated, use``backend='ergast'`` instead
  warnings.warn("Option ``force_ergast`` has been deprecated, use"
d:\Python\Lib\site-packages\fastf1\events.py:498: UserWarning: Option ``force_ergast`` has been deprecated, use``backend='ergast'`` instead
  warnings.warn("Option ``force_ergast`` has been deprecated, use"
core           INFO 	Loading data for Austrian Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Loading data for British Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
core           INFO 	Loading data for Styrian Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for driver_info
core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req          

In [ ]:
# %% [code] ###################################################################
# 7a · CNN-BiLSTM MODEL
class CNNBiLSTM(nn.Module):
    def __init__(self, f=len(FEATURES)):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(f,64,5,padding=2), nn.ReLU(),
            nn.Conv1d(64,128,5,padding=2), nn.ReLU())
        self.rnn = nn.LSTM(128,64,2,batch_first=True,bidirectional=True)
        self.head= nn.Sequential(nn.Linear(128,64), nn.ReLU(),
                                 nn.Linear(64,1), nn.Sigmoid())
    def forward(self,x):                       # x:(B,T,F)
        x = self.conv(x.transpose(1,2)).transpose(1,2)
        _,(h,_) = self.rnn(x)
        h = torch.cat((h[-2],h[-1]),1)
        return self.head(h).squeeze(-1)


In [ ]:
# %% [code] ###################################################################
# 7b · TEMPORAL TRANSFORMER MODEL  (optional – swap into cell 9)
class TimeTransformer(nn.Module):
    def __init__(self, f=len(FEATURES), d=128, heads=4, depth=4):
        super().__init__()
        self.embed = nn.Linear(f,d)
        enc_layer  = nn.TransformerEncoderLayer(d, heads, 256, batch_first=True)
        self.encoder= nn.TransformerEncoder(enc_layer, depth)
        self.cls   = nn.Sequential(nn.Linear(d,1), nn.Sigmoid())
    def forward(self,x):
        z = self.embed(x)
        z = self.encoder(z)
        z = z.mean(1)
        return self.cls(z).squeeze(-1)


In [ ]:
# %% [code] ###################################################################
# 8 · LIGHTNING MODULE
class LitRisk(pl.LightningModule):
    def __init__(self, net):
        super().__init__()
        self.net, self.loss = net, nn.BCELoss()
        self.y_val, self.p_val = None, None
    def forward(self,x): return self.net(x)
    def _step(self,batch,stage):
        x,y = batch
        p = self(x).float(); l = self.loss(p,y)
        self.log(f"{stage}_loss", l, prog_bar=False, logger=True)
        return {"y":y.detach(), "p":p.detach()}
    def training_step(self,batch,idx):   return self._step(batch,"train")
    def validation_step(self,batch,idx): return self._step(batch,"val")
    def validation_epoch_end(self,outs):
        y = torch.cat([o['y'] for o in outs]).cpu().numpy()
        p = torch.cat([o['p'] for o in outs]).cpu().numpy()
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(y,p); self.log("val_auc",auc,prog_bar=True)
        self.y_val, self.p_val = y, p
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=CFG['LR'])


In [ ]:
# %% [code] ###################################################################
# 9 · TRAINING RUN  (swap net = TimeTransformer() if desired)
net = CNNBiLSTM().to(CFG['DEVICE'])
lit = LitRisk(net)
trainer = pl.Trainer(max_epochs=CFG['EPOCHS'],
                     accelerator=CFG['DEVICE'],
                     log_every_n_steps=20)
trainer.fit(lit, train_dl, val_dl)


In [ ]:
# %% [code] ###################################################################
# 10 · METRICS PLOTS
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

y, p = lit.y_val, lit.p_val
fpr, tpr, _     = roc_curve(y, p)
roc_auc         = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(y, p)

fig, ax = plt.subplots(1,3, figsize=(18,4))
ax[0].plot(fpr, tpr); ax[0].plot([0,1],[0,1],'k--')
ax[0].set_title(f"ROC  AUC={roc_auc:.3f}")

ax[1].plot(recall, precision)
ax[1].set_title("Precision-Recall")

th = 0.5
cm = confusion_matrix(y, (p>=th).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["OK","INC"]).plot(ax=ax[2])
ax[2].set_title(f"Confusion @ {th}")
plt.show()


In [ ]:
# %% [code] ###################################################################
# 11 · LEAD-TIME HISTOGRAM
step_s = CFG['STEP_MS']/1000
flag   = (p>=0.5).astype(int)
lead = []
for i in np.where(y==1)[0]:
    j=i
    while j>=0 and flag[j]: j-=1
    lead.append((i-j)*step_s)
plt.hist(lead, bins=20)
plt.title("Warning lead-time distribution (s)")
plt.xlabel("seconds"); plt.ylabel("count")
plt.show()


In [ ]:
# %% [code] ###################################################################
# 12 · GRAD-CAM-STYLE TEMPORAL SALiency
def time_grad(model,x):
    x = x.unsqueeze(0).requires_grad_(True).to(CFG['DEVICE'])
    p = model(x).squeeze(); p.backward()
    g = x.grad.abs().sum(-1).squeeze().cpu().numpy()
    return g / g.max()

sample,_ = val_dl.dataset[0]
imp = time_grad(net, sample)
plt.plot(imp); plt.title("Temporal saliency"); plt.ylabel("relative importance")
plt.show()


In [ ]:
# %% [code] ###################################################################
# 13 · ONNX EXPORT
dummy = torch.randn(1, CFG['WINDOW_MS']//10, len(FEATURES)).to(CFG['DEVICE'])
torch.onnx.export(net, dummy, "risk_model.onnx",
                  input_names=['input'], output_names=['prob'],
                  dynamic_axes={'input':{0:'batch',1:'time'}})
print("✓ risk_model.onnx saved")


In [ ]:
# %% [code] ###################################################################
# 14 · WRITE FastAPI SERVICE  &  PROM METRICS (creates files)
%%bash
mkdir -p driver_api
cat > driver_api/api.py <<'PY'
from fastapi import FastAPI, WebSocket
import onnxruntime as ort, numpy as np, time
from prometheus_client import Counter, Histogram, generate_latest, CONTENT_TYPE_LATEST

FEATURES = ["Throttle","Brake","Steer","Speed","nGear","RPM",
            "GapToLeader","Sector","sin_u","cos_u","DRS","RelSpeedAhead"]
WIN_MS    = 3000
BUF_LEN   = WIN_MS // 10

sess   = ort.InferenceSession("risk_model.onnx", providers=['CPUExecutionProvider'])
buf    = np.zeros((BUF_LEN, len(FEATURES)), dtype=np.float32)
idx    = 0

REQS = Counter("inference_requests_total", "Total predictions")
LAT  = Histogram("inference_latency_seconds", "Latency")

app = FastAPI()

@app.get("/metrics")
def metrics():
    return generate_latest(), 200, {"Content-Type": CONTENT_TYPE_LATEST}

@app.websocket("/telemetry")
async def ws(ws:WebSocket):
    global idx, buf
    await ws.accept()
    while True:
        data = await ws.receive_json()
        buf[idx] = [data[f] for f in FEATURES]
        idx = (idx + 1) % BUF_LEN
        if idx == 0:           # full window → inference
            REQS.inc(); t0=time.time()
            prob = float(sess.run(None, {"input": buf[None]})[0])
            LAT.observe(time.time()-t0)
            await ws.send_json({"risk":prob,"ts":time.time()})
PY


In [ ]:
# %% [code] ###################################################################
# 15 · DOCKERFILE + DOCKER-COMPOSE + PROMETHEUS CONF
%%bash
mkdir -p infra
cat > driver_api/Dockerfile <<'DOCK'
FROM python:3.11-slim
WORKDIR /app
COPY risk_model.onnx ./
COPY api.py ./
RUN pip install fastapi onnxruntime prometheus-client uvicorn[standard]
EXPOSE 8000
CMD ["uvicorn","api:app","--host","0.0.0.0","--port","8000"]
DOCK

cat > infra/prometheus.yml <<'YAML'
global:
  scrape_interval: 5s
scrape_configs:
  - job_name: driver-risk
    metrics_path: /metrics
    static_configs:
      - targets: ['driver-risk:8000']
YAML

cat > docker-compose.yml <<'YAML'
version: "3.9"
services:
  driver-risk:
    build: ./driver_api
    ports: ["8000:8000"]
    restart: unless-stopped
  prometheus:
    image: prom/prometheus:v2.52.0
    volumes: ["./infra/prometheus.yml:/etc/prometheus/prometheus.yml"]
    ports: ["9090:9090"]
  grafana:
    image: grafana/grafana:11.0.0
    ports: ["3000:3000"]
    environment: [GF_SECURITY_ADMIN_PASSWORD=admin]
    depends_on: [prometheus]
YAML


In [ ]:
# %% [code] ###################################################################
# 16-A  · CAN-BUS SAMPLE  (synthetic or replace with real)
import pandas as pd, numpy as np
url = "https://huggingface.co/datasets/motorsport/GT3_CAN/resolve/main/gt3.csv"
df_can = pd.read_csv(url).iloc[:500_000]
SENSORS = df_can.columns[1:]


In [ ]:
# %% [code] ###################################################################
# 16-B  · LSTM-VAE MODEL
class LstmVAE(nn.Module):
    def __init__(self, f=len(SENSORS), h=64, z=16):
        super().__init__()
        self.enc = nn.LSTM(f,h,batch_first=True)
        self.mu, self.lv = nn.Linear(h,z), nn.Linear(h,z)
        self.dec = nn.LSTM(z,h,batch_first=True)
        self.out = nn.Linear(h,f)
    def forward(self,x):
        _,(h,_)=self.enc(x); h=h[-1]
        mu, lv = self.mu(h), self.lv(h)
        z = mu + torch.exp(0.5*lv)*torch.randn_like(mu)
        z = z.unsqueeze(1).repeat(1,x.size(1),1)
        dec,_ = self.dec(z)
        recon = self.out(dec)
        return recon, mu, lv

def vae_loss(x,r,mu,lv):
    rec = (x-r).pow(2).mean()
    kl  = -0.5*torch.mean(1+lv-mu.pow(2)-lv.exp())
    return rec + 0.001*kl, rec


In [ ]:
# %% [code] ###################################################################
# 16-C  · TRAIN VAE
SEQ = 200
def seqs(df):
    a = df[SENSORS].astype(np.float32).values
    return [a[i:i+SEQ] for i in range(0,len(a)-SEQ,SEQ)]
train = torch.tensor(np.stack(seqs(df_can.iloc[:400_000]))).to(CFG['DEVICE'])
val   = torch.tensor(np.stack(seqs(df_can.iloc[400_000:]))).to(CFG['DEVICE'])

vae = LstmVAE().to(CFG['DEVICE'])
opt = torch.optim.Adam(vae.parameters(),1e-3)

for ep in range(5):
    vae.train()
    for i in range(0,len(train),64):
        b = train[i:i+64]
        r,mu,lv = vae(b)
        loss,_ = vae_loss(b,r,mu,lv)
        loss.backward(); opt.step(); opt.zero_grad()
    print(f"ep{ep} loss {loss.item():.4f}")


In [ ]:
# %% [code] ###################################################################
# 16-D  · ANOMALY SCORES + ROC
from sklearn.metrics import roc_auc_score
vae.eval()
def score(t):
    s=[]
    with torch.no_grad():
        for i in range(0,len(t),64):
            b=t[i:i+64]
            r,mu,lv=vae(b); _,rec=vae_loss(b,r,mu,lv); s+=rec.cpu().tolist()
    return np.array(s)

scores_ok   = score(val)
# create simple synthetic faults by adding noise
faulty = val.clone(); faulty += torch.randn_like(faulty)*3
scores_fault= score(faulty)

y_true = np.concatenate([np.zeros_like(scores_ok),
                         np.ones_like(scores_fault)])
y_pred = np.concatenate([scores_ok, scores_fault])
print("ROC-AUC anomaly detector:", roc_auc_score(y_true,y_pred))
